# Utils

Notebook contendo funções reutilizáveis para o pipeline de dados.

In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

# Variáveis Globais
catalog_name = "workspace"
source_volume_path = f"/Volumes/{catalog_name}/bronze/sources/"

def save_delta_table(df, table_name, target_schema, catalog="workspace", mode="overwrite"):
    """
    Salva um DataFrame como uma tabela Delta gerenciada no Unity Catalog.
    """
    full_table_name = f"{catalog}.{target_schema}.{table_name}"
    print(f"Salvando tabela gerenciada: {full_table_name} no modo {mode}...")
    df.write.format("delta") \
        .mode(mode) \
        .option("mergeSchema", "true") \
        .saveAsTable(full_table_name)
    print("Salvo com sucesso!")

def upsert_delta_table(df, table_name, target_schema, join_key, catalog="workspace"):
    """
    Realiza um UPSERT (MERGE) em uma tabela Delta.
    Se a tabela não existir, realiza a carga inicial.
    Suporta chaves compostas (passadas como lista).
    """
    full_table_name = f"{catalog}.{target_schema}.{table_name}"
    
    table_exists = spark.catalog.tableExists(full_table_name)
    
    if not table_exists:
        print(f"Tabela {full_table_name} não existe. Realizando carga inicial (Overwrite)...")
        save_delta_table(df, table_name, target_schema, catalog, mode="overwrite")
    else:
        print(f"Realizando MERGE na tabela: {full_table_name}...")
        target_table = DeltaTable.forName(spark, full_table_name)
        
        # Monta a condição de join
        if isinstance(join_key, list):
            merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in join_key])
        else:
            merge_condition = f"target.{join_key} = source.{join_key}"
        
        target_table.alias("target") \
            .merge(df.alias("source"), merge_condition) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        print("MERGE finalizado com sucesso!")

def optimize_table(table_name, target_schema, zorder_cols=None, catalog="workspace"):
    """
    Executa o comando OPTIMIZE na tabela Delta, opcionalmente aplicando Z-Order.
    """
    full_table_name = f"{catalog}.{target_schema}.{table_name}"
    
    sql_query = f"OPTIMIZE {full_table_name}"
    if zorder_cols:
        cols_str = ", ".join(zorder_cols)
        sql_query += f" ZORDER BY ({cols_str})"
    
    print(f"Executando otimização: {sql_query}...")
    spark.sql(sql_query)
    print("Otimização finalizada com sucesso!")
